# Los genes de PrimeKG integrado son humanos, salvo dos

El grafo de la tesis de origen de PrimeKG integrado identifica sus 17.322 proteinas por Entrez Gene ID (GeneID de NCBI). La tesis no dice en ningun lado que se haya filtrado por especie: lo humano viene de las fuentes que eligio (HIPPIE es "una lista completa, calificada y funcionalmente comentada de las interacciones proteina-proteina del ser humano", seccion 3 de la tesis, y DisGeNET es por definicion de enfermedades humanas), y de SIGNOR solo dice que se usaron "mapeos de vocabulario entre Uniprot y Entrez-ID para obtener enlaces entre complejos y proteinas".

Este notebook comprueba contra NCBI Gene que los genes son humanos y explica, con el codigo de PrimeKG integrado a la vista, que es cada uno de los que no aparecen en la lista humana vigente. Los datos son los que se recibieron de la tesis de origen (`datos/`), su codigo esta clonado en `~/tesis/gcnn_gdas/` (`src/data/network_merging.ipynb` y `src/data/signor_entrez_map.py`), y las dos consultas a NCBI se guardan en `genes_humanos/` para que el notebook se pueda volver a correr sin red.

In [1]:
import json
import os

import pandas as pd
import requests

DATOS = "datos"
CACHE = "genes_humanos"
os.makedirs(CACHE, exist_ok=True)

nodos = pd.read_csv(f"{DATOS}/processed/merged_nodes.csv", dtype={"node_id": str})
#los 421 complejos de SIGNOR comparten el tipo gene_protein pero su id no es un Entrez
genes = nodos[(nodos.node_type == "gene_protein") & ~nodos.node_id.str.startswith("SIGNOR")]
print(len(genes), "genes")
genes.node_source.value_counts()

17322 genes


node_source
disgenet        9689
hippie          6273
primekg_NCBI    1350
signor            10
Name: count, dtype: int64

El `node_source` dice por que fuente entro cada gen al grafo (la primera en la que el pipeline de origen lo encontro, en el orden en que concatena las tablas): DisGeNET, HIPPIE, PrimeKG (que trae los genes de los pathways de Reactome con id de NCBI) y SIGNOR.

## 1. Contra la lista vigente de genes humanos de NCBI

NCBI publica `Homo_sapiens.gene_info`, la lista de todos los GeneID humanos vigentes. Se baja una vez y se guarda con la fecha que informa el servidor.

In [2]:
URL_GENE_INFO = "https://ftp.ncbi.nlm.nih.gov/gene/DATA/GENE_INFO/Mammalia/Homo_sapiens.gene_info.gz"
archivo_gene_info = f"{CACHE}/Homo_sapiens.gene_info.gz"
if not os.path.exists(archivo_gene_info):
    respuesta = requests.get(URL_GENE_INFO, timeout=300)
    respuesta.raise_for_status()
    with open(archivo_gene_info, "wb") as archivo:
        archivo.write(respuesta.content)
    with open(f"{CACHE}/Homo_sapiens.gene_info.FUENTE.txt", "w") as archivo:
        archivo.write(f"{URL_GENE_INFO}\nLast-Modified: {respuesta.headers.get('Last-Modified', '')}\n")
print(open(f"{CACHE}/Homo_sapiens.gene_info.FUENTE.txt").read())

gene_info = pd.read_csv(archivo_gene_info, sep="\t", dtype=str, usecols=["#tax_id", "GeneID", "Symbol"])
humanos_vigentes = set(gene_info.GeneID)
genes = genes.assign(en_lista_vigente=genes.node_id.isin(humanos_vigentes))
print(len(humanos_vigentes), "GeneID humanos vigentes")
print(genes.en_lista_vigente.sum(), "genes de PrimeKG integrado en la lista,", (~genes.en_lista_vigente).sum(), "fuera")

https://ftp.ncbi.nlm.nih.gov/gene/DATA/GENE_INFO/Mammalia/Homo_sapiens.gene_info.gz
Last-Modified: Wed, 16 Sep 2026 16:04:06 GMT

193893 GeneID humanos vigentes
17271 genes de PrimeKG integrado en la lista, 51 fuera


## 2. Los que no estan en la lista: organismo y estado segun NCBI

Que un GeneID no este en la lista vigente puede significar dos cosas: que no es humano, o que es un id humano viejo que NCBI reemplazo por otro o dio de baja. E-utilities (`esummary`) devuelve para cada id el organismo, el estado y, si fue reemplazado, el id actual. Un solo pedido con los 51.

In [3]:
URL_ESUMMARY = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
fuera = genes[~genes.en_lista_vigente]
archivo_esummary = f"{CACHE}/esummary_fuera_de_lista.json"
if not os.path.exists(archivo_esummary):
    respuesta = requests.get(URL_ESUMMARY, params={"db": "gene", "id": ",".join(fuera.node_id), "retmode": "json"}, timeout=120)
    respuesta.raise_for_status()
    with open(archivo_esummary, "w") as archivo:
        archivo.write(respuesta.text)
resumen = json.load(open(archivo_esummary))["result"]

#status 1 es "secondary" (el id fue absorbido por currentid), 2 es "discontinued", vacio es vigente
ESTADO = {"": "vigente", 1: "reemplazado por otro id", 2: "dado de baja"}
detalle = pd.DataFrame([{
    "geneid": uid,
    "nombre_en_integrado": fuera.set_index("node_id").node_name[uid],
    "fuente_en_integrado": fuera.set_index("node_id").node_source[uid],
    "simbolo_ncbi": resumen[uid].get("name", ""),
    "organismo": resumen[uid].get("organism", {}).get("scientificname", ""),
    "estado": ESTADO[resumen[uid].get("status", "")],
    "reemplazado_por": str(resumen[uid].get("currentid", "") or ""),
} for uid in resumen["uids"]])
detalle.to_csv(f"{CACHE}/genes_fuera_de_lista.tsv", sep="\t", index=False)
detalle.groupby(["organismo", "estado"]).size().rename("genes").reset_index()

,organismo,estado,genes
0,Homo sapiens,dado de baja,13
1,Homo sapiens,reemplazado por otro id,36
2,SARS coronavirus Tor2,vigente,1
3,Severe acute respiratory syndrome coronavirus 2,vigente,1


Los 51 se reparten en 49 humanos con id viejo y 2 proteinas virales: la Spike de SARS-CoV (el de 2003, cepa Tor2) y la Spike de SARS-CoV-2. Lo que sigue explica de donde salio cada grupo.

## 3. Los 49 ids humanos viejos: vienen tal cual de los archivos de HIPPIE y DisGeNET

El pipeline de origen no actualiza ni valida los GeneID: toma la columna de ids del archivo de cada fuente y la usa como `node_id`. En `network_merging.ipynb`, la celda que procesa HIPPIE (la que empieza con `hippie = hippie.dropna()`) filtra por score mayor a 0,73, saca el sufijo `_HUMAN` de los nombres y elimina los bucles, y la siguiente arma `hippie_node_data` con `x_id` e `y_id` sin tocarlos. Con DisGeNET pasa lo mismo con `geneId`. Los archivos son de marzo de 2022 (fecha de acceso que da la tesis en las referencias de HIPPIE y DisGeNET), y desde entonces NCBI fusiono o retiro algunos ids. Se comprueba que los 49 estan literalmente en esos archivos.

In [4]:
hippie = pd.read_csv(f"{DATOS}/external/hippie_current.txt", sep="\t", header=None, dtype=str)
ids_hippie = set(hippie[1]) | set(hippie[3])
#el archivo de DisGeNET trae los geneId con espacios adelante, se limpian antes de comparar
disgenet = pd.read_csv(f"{DATOS}/external/curated_gene_disease_associations.tsv", sep="\t", dtype=str)
ids_disgenet = set(disgenet.geneId.str.strip())

viejos = detalle[detalle.organismo == "Homo sapiens"].copy()
viejos["en_archivo_de_su_fuente"] = [
    g in ids_hippie if f == "hippie" else g in ids_disgenet if f == "disgenet" else None
    for g, f in zip(viejos.geneid, viejos.fuente_en_integrado)]
print(viejos.groupby(["fuente_en_integrado", "en_archivo_de_su_fuente"]).size())
viejos[["geneid", "nombre_en_integrado", "fuente_en_integrado", "simbolo_ncbi", "estado", "reemplazado_por"]].sort_values("fuente_en_integrado").reset_index(drop=True)

fuente_en_integrado  en_archivo_de_su_fuente
disgenet             True                        7
hippie               True                       42
dtype: int64


,geneid,nombre_en_integrado,fuente_en_integrado,simbolo_ncbi,estado,reemplazado_por
0,692222,OPA5,disgenet,OPA5,reemplazado por otro id,10059
1,103753527,SCA37,disgenet,SCA37,reemplazado por otro id,1600
2,338435,SCA25,disgenet,SCA25,reemplazado por otro id,87178
3,3653,IPW,disgenet,IPW,reemplazado por otro id,104472715
4,619509,WTRS,disgenet,WTRS,reemplazado por otro id,7467
5,4397,MS,disgenet,MS,reemplazado por otro id,3123
6,54719,RNANC,disgenet,RNANC,reemplazado por otro id,220202
7,643314,K0754,hippie,KIAA0754,reemplazado por otro id,23499
8,727738,AREG,hippie,AREGB,reemplazado por otro id,374
9,653111,TRI49,hippie,TRIM49L2,reemplazado por otro id,642612


Los 42 de HIPPIE y los 7 de DisGeNET estan en los archivos crudos con ese mismo id. Son humanos: 36 fueron absorbidos por otro GeneID (OCLN 4950 paso a 100506658, MICA 4276 a 100507436, ANXA8L2 244 a 728113) y 13 fueron dados de baja, casi todos `LOC...` de HIPPIE, ids provisorios de loci que NCBI despues retiro. Para el grafo no cambia nada: siguen siendo proteinas humanas con sus aristas, solo que con un id que hoy NCBI ya no usa.

## 4. Los 2 virales: entraron por los complejos de SIGNOR, que el pipeline de origen no filtra por especie

SIGNOR es una base de relaciones causales entre entidades, con una columna `TAX_ID` que dice en que organismo se midio la relacion. El pipeline la usa solo para los complejos proteicos. Dos pasos de su codigo dejan pasar proteinas no humanas:

- `src/data/signor_entrez_map.py` toma todos los ids de UniProt de `signor_all_data.tsv` (`signor[signor.DATABASEA == "UNIPROT"].IDA` y lo mismo para B) y los manda al servicio de mapeo de UniProt a Entrez, sin mirar `TAX_ID` ni el organismo de la entrada.
- En `network_merging.ipynb`, la celda que arma `signor_complex_edges` toma toda fila con `TYPEA == "protein"`, `TYPEB == "complex"` y `EFFECT == "form complex"`, y de ahi salen los nodos con `node_source = signor`. Tampoco filtra por `TAX_ID`.

Como SIGNOR registra las interacciones del virus con proteinas humanas, las dos Spike aparecen como componentes de tres complejos: Spike con ACE2 (SARS-CoV), Spike con ACE2 y Spike con NRP1 (SARS-CoV-2).

In [5]:
signor = pd.read_csv(f"{DATOS}/interim/signor_mapped.csv", index_col=0, low_memory=False)
virales = detalle[detalle.organismo != "Homo sapiens"]
entrez_virales = virales.geneid.astype(int).tolist()

#las filas form complex son las que su codigo convierte en aristas, se muestran las de las dos Spike
forma_complejo = signor[(signor.TYPEA == "protein") & (signor.TYPEB == "complex") & (signor.EFFECT == "form complex")]
filas_spike = forma_complejo[forma_complejo.ENTREZ_ID_A.isin(entrez_virales)]
filas_spike[["ENTITYA", "IDA", "ENTREZ_ID_A", "ENTITYB", "IDB", "TAX_ID"]].drop_duplicates().reset_index(drop=True)

,ENTITYA,IDA,ENTREZ_ID_A,ENTITYB,IDB,TAX_ID
0,S,P59594,1489668.0,Spike protein-ACE2,SIGNOR-C240,9534.0
1,S,P0DTC2,43740568.0,CoV2 Spike protein-ACE2,SIGNOR-C254,9534.0
2,S,P0DTC2,43740568.0,CoV2 spike protein-NRP1,SIGNOR-C267,9606.0


In [6]:
#los tres complejos y sus aristas form_complex estan en el grafo final de PrimeKG integrado
aristas = pd.read_csv(f"{DATOS}/processed/merged_edges.csv", dtype=str)
complejos_spike = set(filas_spike.IDB)
print(nodos[nodos.node_id.isin(complejos_spike)][["node_id", "node_name", "node_type", "node_source"]].to_string(index=False))
en_grafo = aristas[(aristas.edge_type == "form_complex") & (aristas.x_id.isin(complejos_spike) | aristas.y_id.isin(complejos_spike))]
print(len(en_grafo) // 2, "aristas form_complex hacia esos complejos (cada una guardada en los dos sentidos)")

    node_id               node_name    node_type node_source
SIGNOR-C240      Spike protein-ACE2 gene_protein      signor
SIGNOR-C254 CoV2 Spike protein-ACE2 gene_protein      signor
SIGNOR-C267 CoV2 spike protein-NRP1 gene_protein      signor
6 aristas form_complex hacia esos complejos (cada una guardada en los dos sentidos)


Un detalle que confunde: el `TAX_ID` de esas filas es 9534 (Chlorocebus, el mono de las celulas Vero donde se midio) o 9606, no el del virus. `TAX_ID` en SIGNOR es el organismo del experimento, no el de la proteina, asi que filtrar por `TAX_ID == 9606` tampoco habria sacado a las Spike. La forma de detectarlas es la que usa este notebook: el organismo de la entrada en UniProt o en NCBI Gene.

Con esto, tambien se explica por que las filas de SIGNOR medidas en raton o rata (hay 2.176 y 543 en `signor_mapped.csv`) no metieron genes de raton: las proteinas de esas filas son humanas, medidas en modelos animales.

## 5. Conclusion

De los 17.322 genes de PrimeKG integrado, 17.320 son humanos segun NCBI Gene (17.271 con id vigente y 49 con un id que NCBI reemplazo o dio de baja despues de 2022, tomados tal cual de los archivos de HIPPIE y DisGeNET). Los otros 2 son las proteinas Spike de SARS-CoV y SARS-CoV-2, que entraron como componentes de tres complejos de SIGNOR porque su codigo no filtra por especie en ese paso. La tesis no lo menciona porque nunca filtro por especie: lo humano es una propiedad de las fuentes, no una decision del pipeline.

In [7]:
resumen_final = pd.DataFrame({
    "grupo": ["humano, id vigente", "humano, id reemplazado por otro", "humano, id dado de baja", "viral (Spike de SARS-CoV y SARS-CoV-2)"],
    "genes": [int(genes.en_lista_vigente.sum()),
              int((detalle.estado == "reemplazado por otro id").sum()),
              int((detalle.estado == "dado de baja").sum()),
              int((detalle.organismo != "Homo sapiens").sum())],
})
assert resumen_final.genes.sum() == len(genes)
resumen_final

,grupo,genes
0,"humano, id vigente",17271
1,"humano, id reemplazado por otro",36
2,"humano, id dado de baja",13
3,viral (Spike de SARS-CoV y SARS-CoV-2),2


## 6. Lo que dicen las fuentes de si mismas

Las citas textuales, de la tesis de origen (seccion 3.3 y 3.4 y sus referencias) y de los papers de HIPPIE, DisGeNET, PrimeKG y SIGNOR, estan reunidas en `genes_humanos/citas_fuentes_humanas.md`. En corto: HIPPIE se define como el interactoma humano, DisGeNET como catalogo de genes asociados a enfermedades humanas, PrimeKG integra Entrez Gene y la red PPI humana, y SIGNOR dice que todas sus interacciones estan mapeadas al proteoma humano pero incluye la red de perturbaciones por SARS-CoV-2, que es por donde entran las dos Spike. El organismo que SIGNOR anota en cada relacion es el del sistema experimental, no el de la proteina.